# Data Import and Preprocessing

Import files and preprocess.  
- Import two datasets
  - The data to use for train/test/validation
  - A dataset which is the "holdout" and contains unseen codes
- Create a data processing pipeline
  - Scale numerics when needed
  - Encode categoricals via a custom data transformer that maps unseen values to 0

In [6]:
# See the Readme.md - you need a .env file to define your paths
import os
from dotenv import load_dotenv
load_dotenv()
input_dir = os.getenv("INPUT_DIR")
output_dir = os.getenv("OUTPUT_DIR")
print(f'Input datasets are in {input_dir}\noutputs written to {output_dir}')

Input datasets are in /Users/valeriecarey/Documents/projects/2026_09_categoricals/data_out/2026_10_14
outputs written to /Users/valeriecarey/Documents/projects/2026_09_categoricals/data_out/2026_11_07


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
import urmc_nn_asgn.config as config

In [25]:
from urmc_nn_asgn.pipeline import build_nn_preprocessing_pipeline

In [27]:
from sklearn.model_selection import train_test_split

In [42]:
import json
import joblib

In [28]:
import importlib
importlib.reload(config)

<module 'urmc_nn_asgn.config' from '/Users/valeriecarey/Documents/projects/2026_09_categoricals/code_nn/urmc_nn_asgn/config.py'>

In [36]:
import urmc_nn_asgn.pipeline  as pipeline
importlib.reload(pipeline)
from urmc_nn_asgn.pipeline import build_nn_preprocessing_pipeline

## Import Data

Data contains simulated inpatient visits, based on 2023 summary statistics and ICD10 frequencies from [HEALTHCARE COST & UTILIZATION PROJECT](https://hcup-us.ahrq.gov/nisoverview.jsp).  Some ICD10 codes and categories were split off into a "holdout" dataset, which you will use to assess the performance of your model on codes not seen in training.  You never train on the holdout data; it's for validation only.

In [10]:
visit_df = pd.read_parquet(Path(input_dir).joinpath('visit_training_data.parquet'))
holdout_df = pd.read_parquet(Path(input_dir).joinpath('visit_holdout_data.parquet'))

In [11]:
visit_df.columns

Index(['VISIT_ID', 'icd10_cm_code', 'ccsr_code', 'FEMALE', 'AWEEKEND',
       'ELECTIVE', 'HOSP_CONTROL_For_profit', 'HOSP_CONTROL_Nonprofit',
       'HOSP_CONTROL_Public', 'AGE', 'LOS', 'visit_cost'],
      dtype='object')

The file [config.py](urmc_nn_asgn/config.py) specifies which columns are numeric, binary, categorical, etc.  `visit_cost` is our target/outcome.  `icd10_cm_code` is our high cardinality categorical.  `ccsr_code` groups these icd10 codes into more general categories; `ccsr_code` is a slightly lower cardinality categorical. Read more about CCSR codes at [AHRQ.gov](https://hcup-us.ahrq.gov/toolssoftware/ccsr/ccs_refined.jsp)

In [19]:
print(f'The ID column(s) in the dataset are [{config.DATA_COLS_ID}].\n',
      f'Binary predictor column(s) are [{config.DATA_COLS_BINARY}], ',
      f'continuous numeric(s) are [{config.DATA_COLS_CONT}],',
      f'categorical predictors are [{config.DATA_COLS_CAT}].'
      f'  Use these to predict {config.DATA_COL_TARGET}')

The ID column(s) in the dataset are [['VISIT_ID']].
 Binary predictor column(s) are [['FEMALE', 'AWEEKEND', 'ELECTIVE', 'HOSP_CONTROL_For_profit', 'HOSP_CONTROL_Nonprofit', 'HOSP_CONTROL_Public']],  continuous numeric(s) are [['AGE', 'LOS']], categorical predictors are [['icd10_cm_code', 'ccsr_code']].  Use these to predict visit_cost


## Train/Test/Eval Split
This is done on the "main" data (visit_df)

In [29]:
X = visit_df[config.DATA_COLS_FEATURES]
y = visit_df[config.DATA_COL_TARGET]

In [32]:
X_train, X_test_eval, y_train, y_test_eval = train_test_split(X, y, 
                                                              test_size=0.30, 
                                                              random_state=54654)

In [33]:
X_eval, X_test, y_eval, y_test = train_test_split(X_test_eval, y_test_eval, 
                                                  test_size=0.50, random_state=23)

## Data Processing Pipeline

In [37]:
pipeline = build_nn_preprocessing_pipeline(encode_missing_unknown="unknown_only")

In [38]:
X_train_trans = pipeline.fit_transform(X_train)

In [55]:
X_eval_trans = pipeline.transform(X_eval)
X_test_trans = pipeline.transform(X_test)

#### Holdout
Also transform the holdout - we will perform inference on this dataset

In [40]:
X_holdout = holdout_df[config.DATA_COLS_FEATURES]
y_holdout = holdout_df[config.DATA_COL_TARGET]

In [41]:
X_holdout_trans = pipeline.transform(X_holdout)

## Save Data, Pipeline and Information

In [47]:
# Dictionary mapping categorical column -> required embedding vocabulary size
fitted_encoder = pipeline.named_transformers_["cat"].named_steps["ordinal_nn"]
embedding_sizes = {
    col: n_cats
    for col, n_cats in zip(config.DATA_COLS_CAT, fitted_encoder.n_categories_)
}
print(embedding_sizes)

{'icd10_cm_code': np.int64(7632), 'ccsr_code': np.int64(421)}


In [51]:
# Save these sizes
json_metadata = {
    "embedding_sizes": {col: int(size) for col, size in embedding_sizes.items()}}

with open(Path(output_dir).joinpath("01_embedding_sizes.json"), "w") as f:
    json.dump(json_metadata, f, indent=4)

In [52]:
# Save the pipeline
joblib.dump(pipeline, Path(output_dir).joinpath("01_fitted_pipeline.json"))

['/Users/valeriecarey/Documents/projects/2026_09_categoricals/data_out/2026_11_07/01_fitted_pipeline.json']

In [58]:
# Save features
np.save(Path(output_dir).joinpath("01_X_train.npy"), X_train_trans)
np.save(Path(output_dir).joinpath("01_X_eval.npy"), X_eval_trans)
np.save(Path(output_dir).joinpath("01_X_test.npy"), X_test_trans)
np.save(Path(output_dir).joinpath("01_X_holdout.npy"), X_holdout_trans)

# Save targets
np.save(Path(output_dir).joinpath("01_y_train.npy"), y_train)
np.save(Path(output_dir).joinpath("01_y_eval.npy"), y_eval)
np.save(Path(output_dir).joinpath("01_y_test.npy"), y_test)
np.save(Path(output_dir).joinpath("01_y_holdout.npy"), y_holdout)